# Load libraries

In [2]:
# !pip install scikit-learn
# !pip install nltk
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 5.8 MB/s  0:00:00


In [3]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
import emoji

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/bualoydgreat/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/bualoydgreat/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/bualoydgreat/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# Load data

In [4]:
# Load the datasets
rappler_docs = pd.read_excel('rappler_corpus.xlsx')
youtube_docs = pd.read_excel('youtube_corpus.xlsx')

# Standardize date_published column
# Rappler docs come in +08:00 timezone
rappler_docs['date_published'] = pd.to_datetime(
    rappler_docs['date_published']).dt.tz_convert(tz='Asia/Manila').dt.tz_localize(None)
# Youtube docs come in UTC timezone
youtube_docs['date_published'] = pd.to_datetime(
    youtube_docs['date_published']).dt.tz_convert(tz='Asia/Manila').dt.tz_localize(None)

# Assign source to each dataset
rappler_docs['source'] = 'rappler'
youtube_docs['source'] = 'youtube'

# Combine the datasets
corpus = pd.concat([
  rappler_docs,
  youtube_docs
], ignore_index=True, axis=0)

corpus

,Unnamed: 0,title,date_published,link,text,tags,source,like_count,reply_parent_id
0,0,"The big reset: PH, India ramp up security ties",2025-08-19 08:00:00,https://www.rappler.com/plus-membership-progra...,Everything about India is huge. Here are some ...,"['India', 'maritime security', 'Philippines-In...",rappler,NaN,NaN
1,1,"View from Manila: Lies, propaganda after Chine...",2025-08-18 19:50:08,https://www.rappler.com/philippines/view-manil...,"MANILA, Philippines – On August 15, days after...","['Philippines-China relations', 'Tess Lazaro',...",rappler,NaN,NaN
2,2,"PH National Maritime Council holds briefing, p...",2025-08-18 14:11:45,https://www.rappler.com/philippines/video-nati...,"MANILA, Philippines – The National Maritime Co...","['maritime security', 'Philippine Coast Guard'...",rappler,NaN,NaN
3,3,"In Beijing’s la-la land, Manila at fault for c...",2025-08-16 08:00:00,https://www.rappler.com/newsbreak/inside-track...,"In Beijing’s imagination, nothing huge happene...","['Philippine Coast Guard', 'West Philippine Sea']",rappler,NaN,NaN
4,4,China accuses Philippine vessels of 'dangerous...,2025-08-15 16:47:05,https://www.rappler.com/philippines/china-stat...,"BEIJING, China – China’s defense ministry accu...","['Hague ruling on West Philippine Sea', 'Scarb...",rappler,NaN,NaN
...,...,...,...,...,...,...,...,...,...
12451,763,One Child Policy ended the PLA years ago. This...,2025-02-19 16:11:25,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,One Child Policy ended the PLA years ago. This...,NaN,youtube,1.0,NaN
12452,764,the bully in action :),2025-02-19 16:06:36,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,the bully in action :),NaN,youtube,5.0,NaN
12453,765,yes ph again by china bullying more years wait...,2025-02-19 18:56:43,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,yes ph again by china bullying more years wait...,NaN,youtube,0.0,Ugy6HTTbPTOtH8B0ZTl4AaABAg
12454,766,"But, South China Sea belong to China by intern...",2025-02-19 20:00:47,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,"But, South China Sea belong to China by intern...",NaN,youtube,0.0,Ugy6HTTbPTOtH8B0ZTl4AaABAg


# Preprocess text

## Load stopwords

In [6]:
from pandas.errors import EmptyDataError

try:
  basic_stopwords = list(
    # Handle empty data error
    pd.read_csv('basic_stopwords.txt', header=None).values.flatten()
  )
except EmptyDataError:
  basic_stopwords = []

try:
  domain_stopwords = list(
    pd.read_csv('domain_stopwords.txt', header=None).values.flatten()
  )
except EmptyDataError:
  domain_stopwords = []

In [12]:
def preprocess_text(corpus, text_column='text'):
  cleaned_corpus = corpus.copy()

  # Lowercase
  cleaned_corpus['cleaned_text'] = cleaned_corpus[text_column].astype(str).str.lower()

  # Lemmatize (by default, lemmatize nouns)
  # Other options:
  #   'v' for verbs
  #   'a' for adjectives
  #   'r' for adverbs
  #   's' for satellites adjectives (adjectives that appear after verbs)
  lemmatizer = WordNetLemmatizer()
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
      lambda text: ' '.join(
        [lemmatizer.lemmatize(word, pos='n') for word in text.split()]
      )
  )

  # Stemmer
  stemmer = PorterStemmer()
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
      lambda text: ' '.join(
        [stemmer.stem(word) for word in text.split()]
      )
  )

  # Remove NLTK stopwords
  en_stopwords_list = stopwords.words('english')
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [
        word for word in text.split() if word not in en_stopwords_list
      ]
    )
  )

  # Remove basic stopwords
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in basic_stopwords]
    )
  )

  # Remove domain stopwords
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in domain_stopwords]
    )
  )

  # Remove trailing and leading whitespaces
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.strip()

  # Remove non-alphanumeric characters
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.replace(r'\W', ' ', regex=True)

  # Remove numbers
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.replace(r'\d+', ' ', regex=True)

  # Remove emojis using emoji library
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in list(emoji.EMOJI_DATA.keys())]
    )
  )

  return cleaned_corpus['cleaned_text']

In [13]:
corpus['cleaned_text'] = preprocess_text(corpus)

In [14]:
corpus.to_excel('cleaned_corpus.xlsx', index=False)

In [15]:
corpus

,Unnamed: 0,title,date_published,link,text,tags,source,like_count,reply_parent_id,cleaned_text
0,0,"The big reset: PH, India ramp up security ties",2025-08-19 08:00:00,https://www.rappler.com/plus-membership-progra...,Everything about India is huge. Here are some ...,"['India', 'maritime security', 'Philippines-In...",rappler,NaN,NaN,everyth india huge astound facts thus philippi...
1,1,"View from Manila: Lies, propaganda after Chine...",2025-08-18 19:50:08,https://www.rappler.com/philippines/view-manil...,"MANILA, Philippines – On August 15, days after...","['Philippines-China relations', 'Tess Lazaro',...",rappler,NaN,NaN,manila philippin august day two china ship col...
2,2,"PH National Maritime Council holds briefing, p...",2025-08-18 14:11:45,https://www.rappler.com/philippines/video-nati...,"MANILA, Philippines – The National Maritime Co...","['maritime security', 'Philippine Coast Guard'...",rappler,NaN,NaN,manila philippin nation maritim council nmc bo...
3,3,"In Beijing’s la-la land, Manila at fault for c...",2025-08-16 08:00:00,https://www.rappler.com/newsbreak/inside-track...,"In Beijing’s imagination, nothing huge happene...","['Philippine Coast Guard', 'West Philippine Sea']",rappler,NaN,NaN,beijing imagination noth huge happen august de...
4,4,China accuses Philippine vessels of 'dangerous...,2025-08-15 16:47:05,https://www.rappler.com/philippines/china-stat...,"BEIJING, China – China’s defense ministry accu...","['Hague ruling on West Philippine Sea', 'Scarb...",rappler,NaN,NaN,beijing china china defens ministri accus phil...
...,...,...,...,...,...,...,...,...,...,...
12451,763,One Child Policy ended the PLA years ago. This...,2025-02-19 16:11:25,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,One Child Policy ended the PLA years ago. This...,NaN,youtube,1.0,NaN,one child polici end pla year ago thi new colo...
12452,764,the bully in action :),2025-02-19 16:06:36,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,the bully in action :),NaN,youtube,5.0,NaN,bulli action
12453,765,yes ph again by china bullying more years wait...,2025-02-19 18:56:43,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,yes ph again by china bullying more years wait...,NaN,youtube,0.0,Ugy6HTTbPTOtH8B0ZTl4AaABAg,ye ph china bulli year wait later
12454,766,"But, South China Sea belong to China by intern...",2025-02-19 20:00:47,https://www.youtube.com/watch?v=vCmPPMdyvkA&lc...,"But, South China Sea belong to China by intern...",NaN,youtube,0.0,Ugy6HTTbPTOtH8B0ZTl4AaABAg,but south china sea belong china intern law co...
